In [1]:
import os
from dotenv import load_dotenv
from FinMind.data import DataLoader
import pandas as pd

In [9]:
load_dotenv()# 會自動找目前工作目錄或上層的 .env（通常你在專案根目錄開 notebook 最順）
api = DataLoader()
api.login_by_token(api_token=os.environ["FINMIND_API_KEY"])

2026-02-08 22:37:42.728 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-08 22:37:42.790 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


True

In [19]:
df = api.taiwan_stock_trading_daily_report(
                securities_trader_id="1440",
                date="2026-02-05",
            )


2026-02-08 23:16:26.856 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockInfo, data_id: 
2026-02-08 23:16:27.334 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockPrice, data_id: 
2026-02-08 23:16:30.549 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTradingDailyReport, data_id: 


In [20]:
df["buy_amount"] = df["buy"] * df["price"]
df["sell_amount"] = df["sell"] * df["price"]

In [21]:
agg = (
    df.groupby(["date", "stock_id"], as_index=False)
    .agg(
        securities_trader=("securities_trader", "first"),
        securities_trader_id=("securities_trader_id", "first"),
        buy=("buy", "sum"),
        sell=("sell", "sum"),
        buy_amount=("buy_amount", "sum"),
        sell_amount=("sell_amount", "sum")
    )
)
agg["net_buy"] = agg["buy"] - agg["sell"]
agg["net_buy_amount"] = agg["buy_amount"] - agg["sell_amount"]
agg["avg_price"] = agg["net_buy_amount"] / agg["net_buy"]

agg = agg.drop(["buy", "sell", "buy_amount", "sell_amount",], axis=1)

In [22]:
agg

,date,stock_id,securities_trader,securities_trader_id,net_buy,net_buy_amount,avg_price
0,2026-02-05,0050,美林,1440,2000,144050.0,72.025000
1,2026-02-05,0052,美林,1440,1000,42810.0,42.810000
2,2026-02-05,0056,美林,1440,189000,7102060.0,37.577037
3,2026-02-05,00635U,美林,1440,-6000,-319500.0,53.250000
4,2026-02-05,00646,美林,1440,4000,273800.0,68.450000
...,...,...,...,...,...,...,...
1384,2026-02-05,9943,美林,1440,-1000,-59600.0,59.600000
1385,2026-02-05,9945,美林,1440,362000,10527350.0,29.081077
1386,2026-02-05,9946,美林,1440,13000,217600.0,16.738462
1387,2026-02-05,9955,美林,1440,2000,69700.0,34.850000


In [ ]:
df_new = pd.read_parquet("../data/brokers/merrill_1440/240101_to_260101.parquet")
df_new[df_new["stock_id"] == "0050"]

     stock_id        date securities_trader  net_buy  net_buy_amount  \
0        0050  2024-01-17                美林 -1581000    -201588400.0   
1        0051  2024-11-21                美林        0          -550.0   
2        0052  2024-08-09                美林    -2000       -343300.0   
3        0055  2025-12-22                美林    -4000       -129480.0   
4        0056  2024-01-09                美林   -17000       -622330.0   
...       ...         ...               ...      ...             ...   
2125     9951  2024-01-05              美林證券   -12000       -895300.0   
2126     9955  2024-01-02                美林     2000         47450.0   
2127     9958  2024-01-02                美林  -103000     -18505500.0   
2128     9960  2024-01-10              美林證券     2000         54300.0   
2129     9962  2024-01-03              美林證券    -2000        -36500.0   

       avg_price  
0     127.506894  
1           -inf  
2     171.650000  
3      32.370000  
4      36.607647  
...          ...  
21

In [ ]:
class BrokerTradeTracker:

    def __init__(self, securities_trader: str, securities_trader_id: str, stock_id: str) -> None:
        self.securities_trader: str = securities_trader
        self.securities_trader_id: str = securities_trader_id
        self.stock_id = stock_id

    def add_record(self, df: pd.DataFrame) -> None:
        self.date = df["date"]

In [35]:
type(pd.DataFrame(["A", "B", "C"]))

pandas.DataFrame

In [ ]:
res = {}
for sid in df_new:
    if sid not in res.key():
        res["sid"] = [()]

,date,stock_id,securities_trader,securities_trader_id,net_buy,net_buy_amount,avg_price
14876,2024-01-17,0050,美林,1440,-1581000,-201588400.0,127.506894
16313,2024-01-18,0050,美林,1440,-249000,-31905000.0,128.132530
19086,2024-01-22,0050,美林,1440,-100000,-13372300.0,133.723000
25957,2024-01-29,0050,美林,1440,-100000,-13636450.0,136.364500
40811,2024-02-21,0050,美林,1440,-99000,-13991850.0,141.331818
...,...,...,...,...,...,...,...
689922,2025-12-19,0050,美林,1440,496000,30984150.0,62.468044
691232,2025-12-22,0050,美林,1440,16000,1014200.0,63.387500
692537,2025-12-23,0050,美林,1440,3000,190900.0,63.633333
695132,2025-12-26,0050,美林,1440,1000,64400.0,64.400000
